# 07 — ENACT Evaluation (Second Rubric)

Re-scores the **same transcripts** produced by notebook 05 (`reports/<model>/ctsr_eval_transcripts.jsonl`)
with a second, independent rubric: the WHO **ENhancing Assessment of Common Therapeutic factors
(ENACT) v1.0** competency tool (`reports/ENACT_inperson_published_220321.pdf`, Kohrt et al. 2015).
Where CTRS-R measures CBT-specific technique, ENACT measures *common therapeutic factors*
(empathy, rapport, harm assessment, hope, feedback elicitation) — an orthogonal lens that helps
check whether CTRS-R gains reflect real skill rather than judge-specific reward hacking.

## Adaptation for text-based LLM-judge scoring

- **Item 1 (Non-verbal communication & active listening) is dropped** — all of its behaviours
  (eye contact, posture, nodding) are unobservable in a text transcript.
- **Items 2–15 keep the instrument's own 1–4 level scheme:**
  - Level 1 = any unhelpful/harmful behaviour (overrides everything else)
  - Level 2 = no unhelpful behaviour; none or only some basic skills
  - Level 3 = no unhelpful behaviour; **all** basic skills
  - Level 4 = Level 3 **plus** any advanced skill
- In-person-only behaviours inside kept items (facial expression, offering a seat) are treated
  as *not applicable* rather than counting against a level.
- **Total = sum of 14 items, range 14–56.**

The rubric, judge prompt, and scoring loop live in [`src/scripts/enact_eval.py`](../src/scripts/enact_eval.py)
(single source of truth — this notebook imports from it). Results go to **`reports/enact/<model>/enact_eval_scores.csv`**,
a separate tree from the CTRS-R results notebook 06 reads.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src.scripts.enact_eval import DEFAULT_MODELS, ENACT_ITEMS, ITEM_KEYS, ITEM_LABELS

ENACT_DIR = Path("../reports/enact")

print(f"{len(ENACT_ITEMS)} ENACT items (2-15), levels 1-4, total range {len(ITEM_KEYS)}-{len(ITEM_KEYS)*4}")
for item in ENACT_ITEMS:
    print(f"  Item {item['number']:2d}: {item['name']}  "
          f"({len(item['harmful'])} harmful / {len(item['basic'])} basic / {len(item['advanced'])} advanced)")

## Run the scoring

Scoring is a standalone script (async DeepSeek judge, one call per transcript, ~100 calls per model).
Run it **from the repo root** in the background with output redirected straight to a log
(no pipes — they buffer):

```bash
nohup env PYTHONUNBUFFERED=1 python3 -m src.scripts.enact_eval > enact_eval_run.log 2>&1 &
tail -f enact_eval_run.log
```

Models already scored are skipped (delete `reports/enact/<model>/enact_eval_scores.csv` to re-score).
`--models a b c` scores a subset. The cell below runs it inline instead, if you prefer.

In [ ]:
# Inline alternative to the shell command above (uses the notebook's event loop).
# Skips models whose enact_eval_scores.csv already exists.
# from src.scripts.enact_eval import run
# await run(DEFAULT_MODELS)

## Load ENACT Scores

In [ ]:
dfs = {}
for name in DEFAULT_MODELS:
    csv_path = ENACT_DIR / name / "enact_eval_scores.csv"
    if not csv_path.exists():
        print(f"  ✗ {name}: not scored yet ({csv_path})")
        continue
    df = pd.read_csv(csv_path)
    dfs[name] = df
    totals = df[ITEM_KEYS].sum(axis=1, min_count=len(ITEM_KEYS))
    print(f"  ✓ {name}: {len(df)} transcripts, mean total {totals.mean():.2f}/56")

means = {name: df[ITEM_KEYS].mean() for name, df in dfs.items()}
cis = {name: 1.96 * df[ITEM_KEYS].sem() for name, df in dfs.items()}
means_df = pd.DataFrame(means).T
means_df.columns = [l.replace("\n", " ") for l in ITEM_LABELS]
print("\nPer-item mean levels (1-4):")
means_df

## Comparison

Side-by-side charts (per-item bars, totals, head-to-head, KTO-vs-DPO progression)
live in [`08_enact_compare.ipynb`](08_enact_compare.ipynb) — the ENACT counterpart of notebook 06.